# PriorAgent — orbital learning experiment
Offline-compatible archive, explicit boundaries, and observed movement planning.
Local comparisons are not hidden leaderboard scores. No saved game evidence is bundled.


In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/prioragent_reviewed_baseline.py
# =====================================================================
# PriorAgent — the consolidated-prior generic engine (Vesica AI / Shovelcat)
# Distilled from live LS20 solves (levels 1-5 at 13/22, 47/123, 47/73, 50/84,
# 58/96 vs human baselines). The transferable layer only:
#   probe-first  : classify what each action does before planning
#   agent-blob   : find the movable 'self' by frame-diffing, learn its lattice
#   landmarks    : small distinct objects are goals/tools; visit them
#   wind-learning: an action that jumps the blob >2 quanta is a transport;
#                  ban the edge for planning, remember the ride
#   novelty      : state-hash graph; prefer the least-tried action from here;
#                  never loop on a proven no-op
#   budget law   : refills exist; a bar that only drains means actions are the
#                  budget — never spend two on the same failed idea
# =====================================================================
import random
import time
from collections import defaultdict, deque
from typing import Any, Optional

import numpy as np

try:
    from agents.agent import Agent
    from arcengine import FrameData, GameAction, GameState
except ImportError:                      # local practice: the shim provides the same surface
    from arcshim import Agent, FrameData, GameAction, GameState

MOVE_ACTIONS = [GameAction.ACTION1, GameAction.ACTION2,
                GameAction.ACTION3, GameAction.ACTION4]


def _grid(latest_frame: FrameData) -> Optional[np.ndarray]:
    fr = latest_frame.frame
    if not fr:
        return None
    return np.asarray(fr[-1], dtype=np.int16)


def _label(mask: np.ndarray):
    """4-connected components without scipy: returns list of (ys, xs) arrays."""
    h, w = mask.shape
    seen = np.zeros_like(mask, dtype=bool)
    comps = []
    for y in range(h):
        for x in range(w):
            if mask[y, x] and not seen[y, x]:
                q = deque([(y, x)])
                seen[y, x] = True
                ys, xs = [], []
                while q:
                    cy, cx = q.popleft()
                    ys.append(cy); xs.append(cx)
                    for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        ny, nx = cy + dy, cx + dx
                        if 0 <= ny < h and 0 <= nx < w and mask[ny, nx] \
                                and not seen[ny, nx]:
                            seen[ny, nx] = True
                            q.append((ny, nx))
                comps.append((np.array(ys), np.array(xs)))
    return comps


class PriorAgent(Agent):
    """Generic probe-first explorer with landmark navigation."""

    MAX_ACTIONS = 2000

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        random.seed(1729 + (hash(self.game_id) % 100000))
        self.prev_grid: Optional[np.ndarray] = None
        self.prev_action: Optional[int] = None
        self.prev_blob: Optional[tuple] = None
        self.effects: dict = defaultdict(lambda: {"moved": 0, "changed": 0,
                                                  "noop": 0, "dy": [], "dx": []})
        self.visit: dict = defaultdict(int)          # state-hash -> visits
        self.tried: dict = defaultdict(lambda: defaultdict(int))
        self.banned: set = set()                     # (state_hash, action_val)
        self.plan: deque = deque()                   # queued action values
        self.plan_target: Optional[tuple] = None
        self.landmark_seen: dict = defaultdict(int)
        self.quantum: int = 0                        # movement lattice, learned
        self.level_at: int = 0
        self.click_targets: deque = deque()
        self.move_block: dict = defaultdict(int)     # (blob,action) -> blocked count
        self.walkable: set = set()                   # colors proven standable
        self.walls: set = set()                      # colors proven blocking
        self.mysteries: list = []                    # unexplained big changes (from the
                                                     # Learning pool: mystery tracking)
        self.steps: int = 0
        self.self_colors: set = set()                # learned per-game self palette
        self.last_progress: int = 0                  # step of last level-up/reset
        self.click_chain: deque = deque()            # productive-click follow-ups

    # ---------- lifecycle ----------
    def is_done(self, frames, latest_frame) -> bool:
        return latest_frame.state is GameState.WIN

    # ---------- perception ----------
    def _blob(self, g: np.ndarray) -> Optional[tuple]:
        """The movable self. Palette detector FIRST (exact position); frame-diff
        as fallback, using the two-component trick: a move leaves an old-spot and
        a new-spot — the distance between their centroids is the exact quantum
        (the diff-centroid alone halves it: the L S20 lesson, twice-paid)."""
        if self.self_colors:                       # learned palette for THIS game
            m = np.isin(g, sorted(self.self_colors))
            comps = [c for c in _label(m) if 1 <= len(c[0]) <= 300]   # v10: 1-px selves exist
            if len(comps) == 1:
                ys, xs = comps[0]
                self._blob_kind = "palette"
                return (int(ys.mean()), int(xs.mean()))
        m = np.isin(g[:56], [9, 12])
        if m.any():
            for ys, xs in _label(m):
                if len(ys) >= 15:
                    vals = g[ys, xs]
                    if (vals == 12).any():
                        self._blob_kind = "palette"
                        return (int(ys.mean()), int(xs.mean()))
        if self.prev_grid is not None and g.shape == self.prev_grid.shape:
            diff = g != self.prev_grid
            nd = int(diff.sum())
            if 0 < nd <= 200:
                comps = sorted(_label(diff), key=lambda c: -len(c[0]))
                if len(comps) == 1 and nd <= 8:
                    # v10: a 1-quantum move leaves ADJACENT old+new cells that merge into one
                    # component — split it by which pixels became background (left) vs colored (entered)
                    ys, xs = comps[0]
                    vals_u, counts_u = np.unique(g, return_counts=True)
                    bg_u = set(int(v) for v, c in zip(vals_u, counts_u) if c > 900)
                    now_vals = g[ys, xs]
                    new_i = [i for i in range(len(ys)) if int(now_vals[i]) not in bg_u]
                    old_i = [i for i in range(len(ys)) if int(now_vals[i]) in bg_u]
                    if new_i and old_i:
                        c1 = (float(np.mean([ys[i] for i in new_i])), float(np.mean([xs[i] for i in new_i])))
                        c0 = (float(np.mean([ys[i] for i in old_i])), float(np.mean([xs[i] for i in old_i])))
                        d = int(round(max(abs(c0[0] - c1[0]), abs(c0[1] - c1[1]))))
                        if 0 < d <= 10 and self.quantum == 0:
                            self.quantum = d
                        cols = set(int(now_vals[i]) for i in new_i) - bg_u
                        if cols and len(cols) <= 4:
                            self.self_colors |= cols
                        self._blob_kind = "diff-adjacent"
                        return (int(round(c1[0])), int(round(c1[1])))
                if len(comps) >= 2 and len(comps[1][0]) >= 0.4 * len(comps[0][0]):
                    c0 = (comps[0][0].mean(), comps[0][1].mean())
                    c1 = (comps[1][0].mean(), comps[1][1].mean())
                    d = int(round(max(abs(c0[0] - c1[0]), abs(c0[1] - c1[1]))))
                    if 0 < d <= 10 and self.quantum == 0:
                        self.quantum = d          # exact old->new displacement
                    # the NEW spot: whichever component's pixels are non-background now
                    ys, xs = comps[0]
                    # learn the self palette: the moving component's current colors
                    vals, counts = np.unique(g, return_counts=True)
                    bg = set(int(v) for v, c in zip(vals, counts) if c > 900)
                    cols = set(int(v) for v in g[ys, xs]) - bg
                    if cols and len(cols) <= 4:
                        self.self_colors |= cols
                    self._blob_kind = "diff2"
                    return (int(ys.mean()), int(xs.mean()))
                ys, xs = comps[0]
                self._blob_kind = "diff"
                return (int(ys.mean()), int(xs.mean()))
        return None

    def _landmarks(self, g: np.ndarray) -> list:
        """Small rare-color components: goals, stations, rings, keyholes."""
        out = []
        vals, counts = np.unique(g, return_counts=True)
        bg = set(int(v) for v, c in zip(vals, counts) if c > 900)
        for v in vals:
            v = int(v)
            if v in bg:
                continue
            comps = _label(g == v)
            if len(comps) > 4:
                continue          # v10: a color scattered as MANY pieces is texture/walls, not a goal
            for ys, xs in comps:
                if 1 <= len(ys) <= 120:   # v10: a landmark can be a single pixel (keys, doors)
                    out.append((v, int(ys.mean()), int(xs.mean()), len(ys)))
        return out

    def _hash(self, g: np.ndarray) -> int:
        return hash(g.tobytes())

    # ---------- planning ----------
    DIRS = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}

    def _learn_wall(self, g, blob, a):
        """the window just beyond the blob in direction a blocked us: its colors
        (not already proven floor) are walls"""
        q = max(1, self.quantum); r = max(1, q // 2)
        dy, dx = self.DIRS[a]
        y, x = blob[0] + dy * q, blob[1] + dx * q
        h, w = g.shape
        if not (r <= y < h - r and r <= x < w - r): return
        win = g[y - r:y + r + 1, x - r:x + r + 1]
        for v in set(int(t) for t in win.flatten()):
            if v not in self.walkable:
                self.walls.add(v)

    def _learn_floor(self, g_before, new_blob):
        """we now stand where g_before shows the old colors: they were passable"""
        r = max(1, self.quantum // 2)
        y, x = new_blob
        h, w = g_before.shape
        if not (r <= y < h - r and r <= x < w - r): return
        win = g_before[y - r:y + r + 1, x - r:x + r + 1]
        for v in set(int(t) for t in win.flatten()):
            self.walkable.add(v)
            self.walls.discard(v)

    def _passable(self, g, y, x, r) -> bool:
        h, w = g.shape
        if not (r <= y < h - r and r <= x < w - r):
            return False
        if not self.walkable:                          # seed: the global mode color
            vals, counts = np.unique(g, return_counts=True)
            self.walkable.add(int(vals[np.argmax(counts)]))
        win = g[y - r:y + r + 1, x - r:x + r + 1]
        ok = 0; tot = 0
        for v in win.flatten():
            v = int(v); tot += 1
            if v in self.walls: return False
            if v in self.walkable: ok += 1
        return ok >= tot * 0.5                         # unknown colors: allowed if half known-floor

    def _bfs_step(self, g, start, goal, q) -> Optional[int]:
        """First move action value of a shortest lattice path start->goal."""
        r = max(1, q // 2)
        dq = deque([(start, None)])
        seen = {start}
        parent = {}
        deltas = {1: (-q, 0), 2: (q, 0), 3: (0, -q), 4: (0, q)}
        found = None
        while dq:
            p, first = dq.popleft()
            if abs(p[0] - goal[0]) + abs(p[1] - goal[1]) <= q:
                found = first
                break
            for a, (dy, dx) in deltas.items():
                np_ = (p[0] + dy, p[1] + dx)
                if np_ in seen or not self._passable(g, np_[0], np_[1], r):
                    continue
                seen.add(np_)
                dq.append((np_, first if first is not None else a))
        return found

    # ---------- the policy ----------
    def choose_action(self, frames, latest_frame) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            self.plan.clear()
            return GameAction.RESET

        g = _grid(latest_frame)
        if g is None:
            return GameAction.RESET

        # level change: clear per-level state, keep learned action effects
        if latest_frame.levels_completed != self.level_at:
            self.level_at = latest_frame.levels_completed
            self.plan.clear()
            self.visit.clear()
            self.banned.clear()
            self.landmark_seen.clear()
            if hasattr(self, "sweep"): del self.sweep
            self.click_gain = defaultdict(int)
            self.move_block = defaultdict(int)
            self.goal = None
            self.walls = set()
            self.mysteries = []
            self.click_chain.clear()
            self.last_progress = self.steps

        # stuck escape: long time on this level AND we mostly revisit known
        # states -> RESET restarts the level with a clean board (spent
        # resources, dead-end configurations, wasted rings all come back)
        if self.steps - self.last_progress > 300:
            self.last_progress = self.steps
            self.plan.clear()
            self.visit.clear()
            self.landmark_seen.clear()
            self.move_block = defaultdict(int)
            self.goal = None
            self.click_chain.clear()
            if hasattr(self, "sweep"): del self.sweep
            self.prev_grid = None
            self.prev_action = None
            return GameAction.RESET

        hsh = self._hash(g)
        self.visit[hsh] += 1
        self.steps += 1
        if self.steps % 40 == 0 and self.move_block:
            # negative confirmation (Learning pool): walls may be doors later —
            # halve the block evidence periodically so bans get re-tested
            for k in list(self.move_block):
                self.move_block[k] //= 2
        blob = self._blob(g)

        # learn from the previous action's observed effect
        if self.prev_grid is not None and self.prev_action is not None:
            eff = self.effects[self.prev_action]
            mag = int((g != self.prev_grid).sum()) \
                if g.shape == self.prev_grid.shape else 0
            changed = mag > 0
            if not changed:
                eff["noop"] += 1
                if self.prev_action in (1, 2, 3, 4) and self.prev_blob is not None:
                    self.move_block[(self.prev_blob, self.prev_action)] += 1
                    self._learn_wall(g, self.prev_blob, self.prev_action)
                ph = self._hash(self.prev_grid)
                if eff["noop"] > 2 and eff["moved"] == 0 and eff["changed"] == 0:
                    self.banned.add((ph, self.prev_action))
            else:
                eff["changed"] += 1
                if mag > 60 and self.prev_action in (1, 2, 3, 4):
                    # a big change no movement explains: remember WHERE (surprise
                    # priority — predictive validation: encode what defied the guess)
                    diff = (g != self.prev_grid)
                    ys, xs = np.where(diff)
                    self.mysteries.append((int(ys.mean()), int(xs.mean())))
                    self.mysteries = self.mysteries[-6:]
                if self.prev_action in (1, 2, 3, 4) and blob is not None \
                        and self.prev_blob is not None and blob == self.prev_blob:
                    # the move was BLOCKED even though pixels changed (a draining
                    # bar is ambient noise, not progress — the LS20 wall-spam bug)
                    self.move_block[(self.prev_blob, self.prev_action)] += 1
                    self._learn_wall(g, blob, self.prev_action)
                elif self.prev_action in (1, 2, 3, 4) and blob is not None \
                        and self.prev_blob is not None and blob != self.prev_blob:
                    self._learn_floor(self.prev_grid, blob)
                if self.prev_action == 6 and getattr(self, "_last_click", None) \
                        and mag >= 4:                     # ignore 1-2px cursor noise
                    if getattr(self, "click_gain", None) is None:
                        self.click_gain = defaultdict(int)
                    self.click_gain[self._last_click] += 1
                    # chain: a productive click earns an immediate replay of the
                    # same spot, then its 8-lattice neighbors (FT09: placements
                    # live on an 8-spaced grid and repeat until the cell is full)
                    cx, cy = self._last_click
                    if len(self.click_chain) < 12:
                        self.click_chain.append((cx, cy))
                        for ddx, ddy in ((8, 0), (0, 8), (-8, 0), (0, -8)):
                            self.click_chain.append((cx + ddx, cy + ddy))
                if blob is not None and self.prev_blob is not None:
                    dy = blob[0] - self.prev_blob[0]
                    dx = blob[1] - self.prev_blob[1]
                    if dy or dx:
                        eff["moved"] += 1
                        eff["dy"].append(dy); eff["dx"].append(dx)
                        step = max(abs(dy), abs(dx))
                        if 0 < step <= 8 and getattr(self, "_blob_kind", "") == "palette":
                            self.quantum = step   # exact when the palette detector holds

        self.prev_grid = g
        self.prev_blob = blob

        # follow an existing plan while it stays sane
        while self.plan:
            a = self.plan.popleft()
            if blob is not None and 1 <= a <= 4 and self.move_block[(blob, a)] >= 2:
                self.plan.clear(); break             # planned move is proven blocked
            self.prev_action = a
            return GameAction["ACTION%d" % a]

        avail = latest_frame.available_actions or []
        avail_vals = []
        for a in avail:
            try:
                avail_vals.append(int(getattr(a, "value", a)))
            except Exception:
                pass
        if not avail_vals:
            avail_vals = [1, 2, 3, 4, 5, 6]

        # click games: productive-click memory + landmark cycling
        pure_click = 6 in avail_vals and not set(avail_vals) & {1, 2, 3, 4}
        stalled = self.effects.get(self.prev_action, {}).get("noop", 0) > 4 \
            if self.prev_action else False
        # budget law for clicks (v10): clicks that never produce anything are a proven
        # no-op — after 20 dry clicks in a movement game, stop clicking entirely
        # (found by the daily mock practice: the stalled-trigger click loop burned
        # 357/400 steps on a game whose clicks did nothing)
        self.click_n = getattr(self, "click_n", 0)
        _good_any = any(v > 0 for v in getattr(self, "click_gain", {}).values()) \
            if getattr(self, "click_gain", None) else False
        clicks_dead = (not pure_click) and self.click_n >= 20 and not _good_any
        if 6 in avail_vals and not clicks_dead \
                and (pure_click or stalled or random.random() < 0.15):
            self.click_n += 1
            if getattr(self, "click_gain", None) is None:
                self.click_gain = defaultdict(int)
            if not hasattr(self, "sweep"):
                self.sweep = deque((x, y) for y in range(4, 64, 8)
                                   for x in range(4, 64, 8))
            good = [k for k, v in self.click_gain.items() if v > 0]
            if self.click_chain:                         # phase 0: ride the chain
                x, y = self.click_chain.popleft()
            elif self.sweep and (not good or random.random() < 0.6):
                x, y = self.sweep.popleft()              # phase 1: coarse sweep
            elif good:
                bx, by = random.choice(good)             # phase 2: focus near hits
                x = bx + random.choice((-4, -2, 0, 2, 4))
                y = by + random.choice((-4, -2, 0, 2, 4))
            else:
                lms = sorted(self._landmarks(g), key=lambda t: t[3])
                if lms and random.random() < 0.6:
                    v, yy, xx, n = random.choice(lms[:12])
                    x, y = xx, yy
                else:
                    x, y = random.randint(2, 61), random.randint(2, 61)
            act = GameAction.ACTION6
            act.set_data({"x": int(max(0, min(63, x))), "y": int(max(0, min(63, y)))})
            self.prev_action = 6
            self._last_click = (int(x), int(y))
            return act

        # movement games: a PERSISTENT GOAL machine — commit to one landmark,
        # walk to it, bump it (collect/enter/insert are all bumps), exhaust, next.
        if blob is not None and self.quantum:
            move_ok = [a for a in (1, 2, 3, 4) if a in avail_vals
                       and self.move_block[(blob, a)] < 2] \
                or [a for a in (1, 2, 3, 4) if a in avail_vals] or [1, 2, 3, 4]
            lms = self._landmarks(g)
            if not hasattr(self, "goal"): self.goal = None
            # re-locate / validate the current goal
            if self.goal is not None:
                v0, gy, gx = self.goal
                same = [t for t in lms if t[0] == v0]
                if same:
                    same.sort(key=lambda t: abs(t[1] - gy) + abs(t[2] - gx))
                    v0, gy, gx = same[0][0], same[0][1], same[0][2]
                    self.goal = (v0, gy, gx)
                else:
                    self.goal = None
            if self.goal is None:
                fresh = [t for t in lms
                         if self.landmark_seen[(t[0], t[1] // 5, t[2] // 5)] < 4]
                if not fresh and lms:
                    # round over: everything visited once. Start a NEW ROUND —
                    # a different visit order lets two-step effects emerge
                    # (enter the keyhole THEN push the box: the T-K-B anchor)
                    self.landmark_seen.clear()
                    self.move_block = defaultdict(int)
                    fresh = lms[:]
                    random.shuffle(fresh)
                if fresh:
                    def myst_bonus(t):
                        return -2 if any(abs(t[1] - my) + abs(t[2] - mx) < 12
                                         for my, mx in self.mysteries) else 0
                    fresh.sort(key=lambda t: (
                        self.landmark_seen[(t[0], t[1] // 5, t[2] // 5)] + myst_bonus(t),
                        abs(t[1] - blob[0]) + abs(t[2] - blob[1])))
                    v0, gy, gx = fresh[0][0], fresh[0][1], fresh[0][2]
                    self.goal = (v0, gy, gx)
                    self.bumps = 0
            if self.goal is not None:
                v0, gy, gx = self.goal
                gkey = (v0, gy // 5, gx // 5)
                dist = abs(gy - blob[0]) + abs(gx - blob[1])
                if dist <= 2 * self.quantum:
                    if getattr(self, "bumps", 0) >= 4:
                        self.landmark_seen[gkey] = 99   # exhausted
                        self.goal = None
                    else:
                        dy, dx = gy - blob[0], gx - blob[1]
                        a = (1 if dy < 0 else 2) if abs(dy) >= abs(dx) \
                            else (3 if dx < 0 else 4)
                        if a in move_ok:
                            self.bumps = getattr(self, "bumps", 0) + 1
                            self.landmark_seen[gkey] += 1
                            self.prev_action = a
                            return GameAction["ACTION%d" % a]
                        self.landmark_seen[gkey] = 99
                        self.goal = None
                else:
                    a = self._bfs_step(g, blob, (gy, gx), self.quantum)
                    if a is not None and a in move_ok:
                        self.prev_action = a
                        return GameAction["ACTION%d" % a]
                    self.landmark_seen[gkey] += 2       # hard to reach; deprioritize
                    self.goal = None
        # novelty fallback: least-tried available action from this state
        nkey = ("pos", blob) if blob is not None else hsh
        cands = [a for a in avail_vals if 1 <= a <= 5 and (hsh, a) not in self.banned
                 and not (blob is not None and 1 <= a <= 4
                          and self.move_block[(blob, a)] >= 2)]
        if not cands:
            cands = [a for a in avail_vals if 1 <= a <= 5] or \
                    ([6] if 6 in avail_vals else [1])
        if cands == [6]:
            act = GameAction.ACTION6
            act.set_data({"x": random.randint(2, 61), "y": random.randint(2, 61)})
            self.prev_action = 6
            return act
        cands.sort(key=lambda a: (self.tried[nkey][a], random.random()))
        a = cands[0]
        self.tried[nkey][a] += 1
        self.prev_action = a
        return GameAction["ACTION%d" % a]


In [ ]:
%%writefile /kaggle/working/orbital_archive.py
"""Immutable, content-addressed learning bundles in a dedicated SQLite store.

Objects are data, never executable commands. Retrieved bundles are candidates,
not current observations. This does not touch the legacy spiral-drive archive.
"""
import hashlib
import json
from pathlib import Path
import sqlite3
import zlib


def encoded(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False).encode('utf-8')


class OrbitalArchive:
    def __init__(self, path):
        self.path = Path(path).resolve()
        self.path.parent.mkdir(parents=True, exist_ok=True)
        existed = self.path.exists()
        self.db = sqlite3.connect(str(self.path))
        self.db.execute('PRAGMA synchronous=FULL')
        if existed:
            version = self.db.execute('PRAGMA user_version').fetchone()[0]
            if version != 1:
                self.db.close()
                raise ValueError('not a supported orbital archive; existing file untouched')
        else:
            with self.db:
                self.db.executescript('''
                    CREATE TABLE objects(id TEXT PRIMARY KEY, payload BLOB NOT NULL);
                    CREATE TABLE bundles(id TEXT PRIMARY KEY, game_id TEXT NOT NULL,
                        instance_id TEXT NOT NULL, level INTEGER NOT NULL, attempt INTEGER NOT NULL,
                        reason TEXT NOT NULL);
                    CREATE TABLE observations(run_id TEXT NOT NULL, observation_id TEXT NOT NULL,
                        ordinal INTEGER NOT NULL, payload_id TEXT NOT NULL,
                        PRIMARY KEY(run_id, observation_id), UNIQUE(run_id, ordinal));
                    PRAGMA user_version=1;
                ''')

    def put(self, value):
        raw = encoded(value)
        if len(raw) > 64_000_000:
            raise ValueError('archive object exceeds 64 MB bound; nothing discarded')
        key = hashlib.sha256(raw).hexdigest()
        with self.db:
            self.db.execute('INSERT OR IGNORE INTO objects VALUES (?,?)', (key, zlib.compress(raw)))
        return key

    def get(self, key):
        row = self.db.execute('SELECT payload FROM objects WHERE id=?', (key,)).fetchone()
        if row is None:
            raise KeyError(key)
        raw = zlib.decompress(row[0])
        if hashlib.sha256(raw).hexdigest() != key:
            raise ValueError('archive integrity mismatch')
        return json.loads(raw)

    def save_bundle(self, bundle):
        for field in ('game_id', 'instance_id', 'reason'):
            if not isinstance(bundle.get(field), str) or not bundle[field]:
                raise ValueError('bundle identity/reason required')
        for field in ('level', 'attempt'):
            if type(bundle.get(field)) is not int or bundle[field] < 0:
                raise ValueError('nonnegative bundle level/attempt required')
        key = self.put(bundle)
        with self.db:
            self.db.execute('INSERT OR IGNORE INTO bundles VALUES (?,?,?,?,?,?)',
                            (key, bundle['game_id'], bundle['instance_id'], bundle['level'],
                             bundle['attempt'], bundle['reason']))
        return key

    def append_observation(self, run_id, observation_id, ordinal, payload):
        key = self.put(payload)
        previous = self.db.execute('SELECT ordinal,payload_id FROM observations WHERE run_id=? AND observation_id=?',
                                   (run_id, observation_id)).fetchone()
        if previous:
            if previous != (ordinal, key):
                raise ValueError('conflicting replay observation')
            return False
        with self.db:
            self.db.execute('INSERT INTO observations VALUES (?,?,?,?)', (run_id, observation_id, ordinal, key))
        return True

    def observations(self, run_id):
        return [self.get(row[0]) for row in self.db.execute(
            'SELECT payload_id FROM observations WHERE run_id=? ORDER BY ordinal', (run_id,))]

    def retrieve(self, *, game_id=None, limit=8):
        if type(limit) is not int or not 1 <= limit <= 64:
            raise ValueError('retrieval limit must be 1..64')
        if game_id is None:
            rows = self.db.execute('SELECT id FROM bundles ORDER BY rowid DESC LIMIT ?', (limit,))
        else:
            rows = self.db.execute('SELECT id FROM bundles WHERE game_id=? ORDER BY rowid DESC LIMIT ?', (game_id, limit))
        return [dict(bundle_id=row[0], bundle=self.get(row[0]), status='retrieved_candidate',
                     adds_observations=False, execution_authorized=False) for row in rows]

    def close(self):
        self.db.close()


In [ ]:
%%writefile /kaggle/working/learning_session.py
"""Durable boundary-aware learning from observable before/after game frames.

Features are explicitly measured colour presence. Outcome is exact frame change,
not movement, progress or causality. No archive command is executed on retrieval.
"""
from copy import deepcopy
import hashlib
import json

from automatic_pin_refiner import AutomaticPinRefiner
from evidence_pin import EvidencePin
from orbital_archive import encoded


STATES = {'NOT_PLAYED', 'NOT_FINISHED', 'GAME_OVER', 'WIN'}


def validate_observation(record):
    for key in ('observation_id', 'game_id', 'instance_id'):
        if not isinstance(record.get(key), str) or not record[key] or len(record[key]) > 256:
            raise ValueError('named observation, game and instance IDs required')
    if type(record.get('level')) is not int or record['level'] < 0 or record.get('state') not in STATES:
        raise ValueError('explicit valid level and engine state required')
    action = record.get('previous_action')
    if action is not None and (type(action) is not int or not 0 <= action <= 7):
        raise ValueError('explicit previous action 0..7 or None required')
    frame = record.get('frame')
    if not isinstance(frame, list) or not 1 <= len(frame) <= 128 or not isinstance(frame[0], list) or not 1 <= len(frame[0]) <= 128:
        raise ValueError('bounded rectangular ARC frame required')
    if any(not isinstance(row, list) or len(row) != len(frame[0]) or any(type(v) is not int or not 0 <= v <= 15 for v in row) for row in frame):
        raise ValueError('rectangular integer palette 0..15 required')
    return deepcopy(record)


def classify_boundary(previous, current):
    if previous is None:
        return 'bootstrap'
    if current['game_id'] != previous['game_id']:
        return 'game_change'
    if current['instance_id'] != previous['instance_id']:
        return 'session_change'
    if current['previous_action'] == 0:
        return 'reset'
    if current['level'] != previous['level']:
        return 'level_change'
    if current['state'] == 'GAME_OVER' and previous['state'] != 'GAME_OVER':
        return 'death'
    if current['state'] == 'WIN' and previous['state'] != 'WIN':
        return 'win'
    if (len(current['frame']), len(current['frame'][0])) != (len(previous['frame']), len(previous['frame'][0])):
        return 'uncertain_boundary'
    if previous['state'] in ('GAME_OVER', 'WIN') and current['state'] == 'NOT_FINISHED':
        return 'uncertain_boundary'  # Never invent an unobserved reset action.
    return 'transition'


class LearningSession:
    def __init__(self, archive, run_id):
        if not isinstance(run_id, str) or not run_id:
            raise ValueError('run ID required')
        self.archive, self.run_id = archive, run_id
        self.previous, self.attempt = None, 0
        self.pins, self.roles, self.commands, self.refinements = {}, {}, [], []
        self.closed_bundles, self.seen, self.events = [], {}, []
        self.fault, self.terminal_archived = False, False
        self.refiner = AutomaticPinRefiner()

    @classmethod
    def resume(cls, archive, run_id):
        session = cls(archive, run_id)
        for record in archive.observations(run_id):
            session.observe(record, replay=True)
        return session

    def _bundle(self, reason):
        if self.previous is None:
            return None
        return dict(game_id=self.previous['game_id'], instance_id=self.previous['instance_id'],
                    level=self.previous['level'], attempt=self.attempt, run_id=self.run_id,
                    reason=reason, commands=deepcopy(self.commands),
                    pins={str(a): p.snapshot() for a, p in self.pins.items()},
                    roles=deepcopy(self.roles), refinements=deepcopy(self.refinements),
                    last_observation=self.previous['observation_id'],
                    outcome='failed' if reason == 'death' else 'won' if reason == 'win' else 'incomplete',
                    execution_authorized=False)

    def archive_current(self, reason='checkpoint'):
        bundle = self._bundle(reason)
        if bundle is None:
            return None
        key = self.archive.save_bundle(bundle)
        if key not in self.closed_bundles:
            self.closed_bundles.append(key)
        return key

    def observe(self, record, *, replay=False):
        if self.fault:
            raise ValueError('session faulted; explicitly resume durable observations')
        record = validate_observation(record)
        key = record['observation_id']
        digest = hashlib.sha256(encoded(record)).hexdigest()
        if key in self.seen:
            if self.seen[key] != digest:
                raise ValueError('conflicting duplicate observation')
            return dict(status='duplicate', adds_observations=False)
        event = classify_boundary(self.previous, record)
        try:
            # Commit actual observations first; a failure in derivation can be
            # replayed from the durable journal. Never silently retry game actions.
            if not replay:
                self.archive.append_observation(self.run_id, key, len(self.seen), record)
            before = self.previous
            if event in ('game_change', 'session_change', 'reset', 'level_change', 'uncertain_boundary'):
                if not self.terminal_archived:
                    self.archive_current(event)
                if event in ('game_change', 'session_change'):
                    self.attempt = 0
                else:
                    self.attempt += 1
                self.pins, self.roles, self.commands, self.refinements = {}, {}, [], []
                self.terminal_archived = False
            frame_id = self.archive.put(record['frame'])
            # Every command/outcome is retained, but boundary frames are never
            # used as ordinary within-board effect examples.
            if before is not None and record['previous_action'] is not None:
                command = dict(action=record['previous_action'], data=deepcopy(record.get('previous_data', {})),
                               before=before['observation_id'], after=key, event=event,
                               before_frame=self.archive.put(before['frame']), after_frame=frame_id)
                if event in ('transition', 'death', 'win'):
                    self.commands.append(command)
            learned = False
            if event == 'transition' and before is not None and record['previous_action'] not in (None, 0) and before['state'] == record['state'] == 'NOT_FINISHED':
                action = record['previous_action']
                if action not in self.pins:
                    scope = json.dumps([self.run_id, record['game_id'], record['instance_id'], record['level'], self.attempt])
                    self.pins[action] = EvidencePin(f'action-{action}-frame-change', scope=scope,
                        subject='full-observed-frame', action=action, question='Does the returned frame differ?',
                        factors={f'has_colour_{i}': f'palette-{i}' for i in range(16)})
                    self.roles[action] = dict(fit=[], check=[])
                pin, roles = self.pins[action], self.roles[action]
                count = sum(map(len, roles.values()))
                if count >= 512:
                    # Bounded observation windows detach rather than evict.
                    self.archive_current('capacity')
                    old = pin.snapshot()
                    pin = EvidencePin(old['id'], scope=old['scope']+f'/window-{len(self.commands)}',
                                      subject=old['subject'], action=action, question=old['question'], factors=old['factor_colours'])
                    self.pins[action], self.roles[action] = pin, dict(fit=[], check=[])
                    roles, count = self.roles[action], 0
                colours = {v for row in before['frame'] for v in row}
                pin.record(key, scope=pin.snapshot()['scope'], action=action,
                    bits={f'has_colour_{i}': i in colours for i in range(16)},
                    outcome=before['frame'] != record['frame'],
                    observation={'before': {'frame_id': self.archive.put(before['frame'])}, 'after': {'frame_id': frame_id}},
                    route='central-port', direction=[0, 0, 1])
                roles['check' if count % 3 == 2 else 'fit'].append(key)
                if count >= 7 and (count+1) % 16 == 0:
                    result = self.refiner.step(pin, fit_ids=roles['fit'], check_ids=roles['check'])
                    if result['changed']:
                        self.refinements.append(result['proposal'])
                        learned = True
            self.previous = record
            if event in ('death', 'win'):
                self.archive_current(event)
                self.terminal_archived = True
            self.seen[key] = digest
            self.events.append(dict(observation_id=key, event=event, game_id=record['game_id'],
                                    instance_id=record['instance_id'], level=record['level'], attempt=self.attempt))
            return dict(status='observed', event=event, refined=learned,
                        attempt=self.attempt, execution_authorized=False)
        except Exception:
            self.fault = True
            raise

    def snapshot(self):
        return deepcopy(dict(run_id=self.run_id, attempt=self.attempt, events=self.events,
                             pins={a: p.snapshot() for a, p in self.pins.items()}, roles=self.roles,
                             refinements=self.refinements, closed_bundles=self.closed_bundles,
                             fault=self.fault, execution_authorized=False))


In [ ]:
%%writefile /kaggle/working/automatic_pin_refiner.py
"""Conservative automatic boolean splits for EvidencePin. No game execution.

Fit chooses a unique exact split; disjoint checks accept or reject that choice.
This deliberately abstains on noisy, tied, incomplete or multi-step-only fits.
It learns within caller-supplied features, not from raw pixels or vortex physics.
"""
from copy import deepcopy
from hashlib import sha256
import json


def _canonical(value):
    return json.dumps(value, sort_keys=True, allow_nan=False, separators=(',', ':'))


def _fingerprint(state):
    # Delivery angles/counts are metadata, not new evidence. No proposal survives
    # changes to the actual model, identity, feature schema, or trial records.
    fields = ('id', 'scope', 'subject', 'action', 'question', 'revision',
              'factor_colours', 'bands', 'trials', 'refinements')
    return sha256(_canonical({k: state[k] for k in fields}).encode()).hexdigest()


def _ids(values, known):
    if not isinstance(values, (list, tuple)) or any(not isinstance(v, str) for v in values):
        raise ValueError('explicit list/tuple of observation IDs required')
    if len(values) != len(set(values)) or not set(values) <= known:
        raise ValueError('unique known observation IDs required')
    return sorted(values)


def _partition(trials, conditions):
    matching, unknown = [], []
    for trial in trials:
        bits = trial['bits']
        if any(k in bits and bits[k] != v for k, v in conditions.items()):
            continue
        if any(k not in bits for k in conditions):
            unknown.append(trial['id'])
        else:
            matching.append(trial)
    return matching, unknown


class AutomaticPinRefiner:
    def __init__(self, *, minimum_fit=2, minimum_check=2):
        if any(type(v) is not int or v < 1 for v in (minimum_fit, minimum_check)):
            raise ValueError('positive integer per-branch support quotas required')
        self._minimum_fit, self._minimum_check = minimum_fit, minimum_check

    def inspect(self, pin, *, fit_ids, check_ids):
        """Read-only proposal. Checks never select a feature among tied fits.

        Data roles must be assigned by the caller before inspecting outcomes.
        IDs ensure provenance/deduplication, not statistical independence.
        Unassigned trials cannot promote a split, but may veto a contradiction.
        """
        state = pin.snapshot()
        known = {t['id'] for t in state['trials']}
        fit_ids, check_ids = _ids(fit_ids, known), _ids(check_ids, known)
        if set(fit_ids) & set(check_ids):
            raise ValueError('fit and check sets must be disjoint')
        index = {t['id']: t for t in state['trials']}
        fit = [index[k] for k in fit_ids]
        checks = [index[k] for k in check_ids]
        reports, selected = [], None
        for rule in state['bands']:
            if rule['children']:
                continue
            rows, missing_parent = _partition(fit, rule['conditions'])
            validation, missing_check_parent = _partition(checks, rule['conditions'])
            report = dict(rule_id=rule['id'], conditions=deepcopy(rule['conditions']),
                          fit_ids=[t['id'] for t in rows], check_ids=[t['id'] for t in validation],
                          candidates=[], questions=[], status='no_supported_split')
            reports.append(report)

            def question(kind, text, **details):
                report['questions'].append(dict(kind=kind, question=text,
                    scope=state['scope'], subject=state['subject'], action=state['action'],
                    target_question=state['question'], rule_id=rule['id'],
                    conditions=deepcopy(rule['conditions']), execution_ready=False, **details))

            if missing_parent or missing_check_parent:
                report['status'] = 'missing_parent_context'
                question('measure_missing_context', 'Measure missing branch conditions; do not guess them.',
                         observation_ids=sorted(set(missing_parent+missing_check_parent)))
                continue
            if len({t['outcome'] for t in rows}) < 2:
                report['status'] = 'no_observed_contrast'
                # No constant-rule installer yet: a uniform sample cannot choose
                # a distinguishing condition. Existing rules are left intact.
                question('seek_contrast', 'Can a reachable case challenge the current outcome pattern?')
                continue
            for factor in sorted(set(state['factor_colours'])-set(rule['conditions'])):
                candidate = dict(factor=factor, status='unsupported', branches=[])
                report['candidates'].append(candidate)
                missing = [t['id'] for t in rows if factor not in t['bits']]
                if missing:
                    candidate.update(status='missing_fit_feature', missing_ids=missing)
                    continue
                for value in (False, True):
                    branch = [t for t in rows if t['bits'][factor] == value]
                    outcomes = {t['outcome'] for t in branch}
                    candidate['branches'].append(dict(value=value, fit_ids=[t['id'] for t in branch],
                        expected=next(iter(outcomes)) if len(outcomes) == 1 else None))
                if any(len(b['fit_ids']) < self._minimum_fit for b in candidate['branches']):
                    candidate['status'] = 'insufficient_fit_support'
                elif any(b['expected'] is None for b in candidate['branches']):
                    candidate['status'] = 'mixed_branch_outcomes'
                elif candidate['branches'][0]['expected'] == candidate['branches'][1]['expected']:
                    candidate['status'] = 'no_distinguishing_effect'
                else:
                    candidate['status'] = 'fit_supported'
            supported = [c for c in report['candidates'] if c['status'] == 'fit_supported']
            if len(supported) > 1:
                report['status'] = 'ambiguous_fit'
                # A concrete pair of assignments where the first two tied
                # hypotheses disagree; neither outcome is asserted to be true.
                first, second = supported[:2]
                choices = []
                for a in first['branches']:
                    for b in second['branches']:
                        if a['expected'] != b['expected']:
                            choices.append(dict(bits={**rule['conditions'], first['factor']: a['value'],
                                                      second['factor']: b['value']},
                                                predictions={first['factor']: a['expected'],
                                                             second['factor']: b['expected']}))
                question('distinguish_hypotheses', 'Can either disagreement case be reached and observed?',
                         factors=[c['factor'] for c in supported], cases=choices, reachability='unknown')
                continue
            if not supported:
                question('collect_or_expand_features',
                         'Collect branch contrasts or measure another condition; no supported one-step split exists.',
                         candidate_statuses={c['factor']: c['status'] for c in report['candidates']})
                continue
            candidate = supported[0]
            factor = candidate['factor']
            report['chosen_on_fit'] = factor
            unknown = [t['id'] for t in validation if factor not in t['bits']]
            errors = []
            for branch in candidate['branches']:
                matching = [t for t in validation if factor in t['bits'] and t['bits'][factor] == branch['value']]
                branch['check_ids'] = [t['id'] for t in matching]
                errors.extend(t['id'] for t in matching if t['outcome'] != branch['expected'])
            report.update(check_errors=errors, missing_check_feature=unknown)
            if errors:
                report['status'] = 'check_failed'
                question('explain_check_failure', 'Which missing condition explains the checking counterexamples?',
                         observation_ids=errors)
                continue
            if unknown or any(len(b['check_ids']) < self._minimum_check for b in candidate['branches']):
                report['status'] = 'insufficient_check_support'
                question('collect_check_cases', 'Obtain fresh checking trials for both branches, without refitting this choice.',
                         factor=factor, missing_ids=unknown,
                         branches=[dict(value=b['value'], required=max(0, self._minimum_check-len(b['check_ids'])))
                                   for b in candidate['branches']])
                continue
            # Never hide a known counterexample just because the caller did not
            # put it into a fit/check cohort. These records can only veto.
            all_rows, all_unknown = _partition(state['trials'], rule['conditions'])
            all_unknown += [t['id'] for t in all_rows if factor not in t['bits']]
            all_errors = [t['id'] for t in all_rows if factor in t['bits'] and
                          t['outcome'] != candidate['branches'][int(t['bits'][factor])]['expected']]
            if all_unknown or all_errors:
                report['status'] = 'unresolved_retained_evidence'
                question('resolve_retained_evidence', 'Resolve retained counterexamples or missing context before refining.',
                         missing_ids=sorted(set(all_unknown)), counterexample_ids=all_errors)
                continue
            report['status'] = 'ready'
            if selected is None:  # At most one mutation per step; then recompute.
                selected = dict(rule_id=rule['id'], factor=factor,
                                absent_expected=candidate['branches'][0]['expected'],
                                present_expected=candidate['branches'][1]['expected'])
        return dict(schema='automatic-pin-refiner-v1', pin_id=state['id'], scope=state['scope'],
                    revision=state['revision'], evidence_fingerprint=_fingerprint(state),
                    policy=dict(minimum_fit=self._minimum_fit, minimum_check=self._minimum_check),
                    fit_ids=fit_ids, check_ids=check_ids, reports=reports, selected=selected,
                    status='ready' if selected else 'abstain', execution_authorized=False)

    def apply(self, pin, proposal):
        """Revalidate the complete proposal before installing a local candidate.

        Synchronous/single-owner API, not thread-safe or a security boundary.
        Calling this changes only EvidencePin, never a game or active planner.
        """
        if not isinstance(proposal, dict) or 'fit_ids' not in proposal or 'check_ids' not in proposal:
            raise ValueError('refiner proposal required')
        current = self.inspect(pin, fit_ids=proposal['fit_ids'], check_ids=proposal['check_ids'])
        if _canonical(proposal) != _canonical(current):
            raise ValueError('stale, altered, or incompatible proposal')
        if current['status'] != 'ready':
            return dict(status='abstained', changed=False, proposal=current)
        choice = current['selected']
        children = pin.lift(choice['rule_id'], factor=choice['factor'],
                            absent_expected=choice['absent_expected'], present_expected=choice['present_expected'],
                            expected_revision=current['revision'])
        return dict(status='refined_candidate', changed=True, children=list(children),
                    proposal=current, execution_authorized=False)

    def step(self, pin, *, fit_ids, check_ids):
        return self.apply(pin, self.inspect(pin, fit_ids=fit_ids, check_ids=check_ids))


In [ ]:
%%writefile /kaggle/working/evidence_pin.py
"""Persistent question shafts with append-only trials and explicit refinements.

Offline representation, not perception or causal inference. Directions/colours
are display metadata. Every gap is a named test/model question, not empty space.
"""
from copy import deepcopy
import json
from math import isfinite, sqrt


def _name(value):
    if not isinstance(value, str) or not value.strip() or len(value) > 256:
        raise ValueError('nonempty name of at most 256 characters required')
    return value


def _json_copy(value):
    raw = json.dumps(value, allow_nan=False, sort_keys=True)
    if len(raw.encode('utf-8')) > 100_000:
        raise ValueError('observation exceeds pin record budget')
    return json.loads(raw)


def _direction(value):
    if not isinstance(value, (tuple, list)) or len(value) != 3 or any(
            type(v) not in (int, float) or not isfinite(v) for v in value):
        raise ValueError('three finite direction components required')
    size = sqrt(sum(v*v for v in value))
    if not isfinite(size) or size == 0:
        raise ValueError('finite nonzero direction required')
    return [v/size for v in value]


class EvidencePin:
    """One scoped subject/action/outcome question, with a finite boolean schema.

    Callers supply observed bits and before/after data; this class cannot verify
    their origin. A missing bit means unknown, not false. Stable trial IDs are
    deduplicated across eye routes. Split rules are proposed by the caller.
    """

    def __init__(self, pin_id, *, scope, subject, action, question, factors,
                 expected=None, max_trials=512, max_rules=63):
        for value in (pin_id, scope, subject, question):
            _name(value)
        if type(action) is not int:
            raise ValueError('integer action required')
        if expected is not None and type(expected) is not bool:
            raise ValueError('expected must be boolean or unknown')
        if not isinstance(factors, dict) or not 1 <= len(factors) <= 16:
            raise ValueError('declare 1..16 factor names with display colours')
        for name, colour in factors.items():
            _name(name)
            _name(colour)
        if any(type(v) is not int or v < 1 for v in (max_trials, max_rules)):
            raise ValueError('positive record budgets required')
        self._identity = dict(id=pin_id, scope=scope, subject=subject,
                              action=action, question=question)
        self._factors = deepcopy(factors)
        self._limits = (max_trials, max_rules)
        self._trials, self._arrivals, self._events = {}, {}, []
        self._rules = {'root': dict(id='root', parent=None, conditions={},
                                   expected=expected, children=[], depth=0)}
        self._revision = 0

    def record(self, observation_id, *, scope, action, bits, outcome,
               observation, route, direction):
        """Add an actual supplied trial; repeated routing adds no trial weight."""
        _name(observation_id)
        _name(route)
        if scope != self._identity['scope'] or type(action) is not int or action != self._identity['action']:
            raise ValueError('wrong scope/action')
        if type(outcome) is not bool or not isinstance(bits, dict) or any(
                k not in self._factors or type(v) is not bool for k, v in bits.items()):
            raise ValueError('observed boolean outcome and declared boolean bits required')
        if not isinstance(observation, dict) or not {'before', 'after'} <= observation.keys():
            raise ValueError('retain before/after observations')
        trial = dict(id=observation_id, bits=deepcopy(bits), outcome=outcome,
                     observation=_json_copy(observation))
        arrival = dict(route=route, direction=_direction(direction))
        previous = self._trials.get(observation_id)
        if previous is not None and json.dumps(previous, sort_keys=True) != json.dumps(trial, sort_keys=True):
            raise ValueError('conflicting duplicate trial; original retained')
        arrivals = self._arrivals.get(observation_id, [])
        if arrival not in arrivals and len(arrivals) >= 8:
            raise ValueError('arrival metadata budget exhausted')
        if previous is None and len(self._trials) >= self._limits[0]:
            raise ValueError('trial budget exhausted; nothing evicted')
        if previous is None:
            self._trials[observation_id] = trial
        self._arrivals.setdefault(observation_id, [])
        if arrival not in arrivals:
            self._arrivals[observation_id].append(arrival)
        return previous is None

    def lift(self, rule_id, *, factor, absent_expected, present_expected,
             expected_revision):
        """Retain a leaf as parent; add two candidate branches atomically.

        Does not close questions, erase counterexamples, or certify a new rule.
        Historical trials are partitioned by recorded bits, never guessed bits.
        """
        if type(expected_revision) is not int or expected_revision != self._revision:
            raise ValueError('stale model revision')
        if rule_id not in self._rules or factor not in self._factors:
            raise ValueError('unknown rule or factor')
        parent = self._rules[rule_id]
        if parent['children'] or factor in parent['conditions']:
            raise ValueError('split an unsplit leaf on a new condition')
        if any(v is not None and type(v) is not bool for v in (absent_expected, present_expected)):
            raise ValueError('candidate outcomes must be boolean or unknown')
        if len(self._rules)+2 > self._limits[1]:
            raise ValueError('rule budget exhausted; nothing replaced')
        revision = self._revision+1
        children = []
        for value, expected in ((False, absent_expected), (True, present_expected)):
            child_id = f'r{revision}:{int(value)}'
            children.append(dict(id=child_id, parent=rule_id,
                                 conditions={**parent['conditions'], factor: value},
                                 expected=expected, children=[], depth=parent['depth']+1))
        self._events.append(dict(revision=revision, kind='lift', parent=rule_id,
                                 previous_rule=deepcopy(parent), factor=factor,
                                 children=[c['id'] for c in children]))
        for child in children:
            self._rules[child['id']] = child
        parent['children'] = [c['id'] for c in children]
        self._revision = revision
        return tuple(parent['children'])

    def _partition(self, conditions):
        matching, unknown = [], []
        for trial in self._trials.values():
            bits = trial['bits']
            if any(k in bits and bits[k] != v for k, v in conditions.items()):
                continue
            if any(k not in bits for k in conditions):
                unknown.append(trial['id'])
            else:
                matching.append(trial)
        return matching, unknown

    def predict(self, bits):
        """Read a candidate leaf prediction; not permission or confidence."""
        if not isinstance(bits, dict) or any(k not in self._factors or type(v) is not bool for k, v in bits.items()):
            raise ValueError('declared boolean bits required')
        for rule in self.snapshot()['bands']:
            if rule['children'] or not all(k in bits and bits[k] == v for k, v in rule['conditions'].items()):
                continue
            if rule['expected'] is None or not rule['trial_ids'] or rule['counterexample_ids']:
                break
            return dict(outcome=rule['expected'], rule_id=rule['id'], status='candidate',
                        execution_authorized=False)
        return dict(outcome=None, rule_id=None, status='abstain', execution_authorized=False)

    def snapshot(self):
        """Height is layout only: old IDs/evidence survive insertion below them."""
        bands, gaps = [], []
        max_depth = max(r['depth'] for r in self._rules.values())
        for rule in self._rules.values():
            trials, unknown = self._partition(rule['conditions'])
            outcomes = {t['outcome'] for t in trials}
            status = 'unknown' if not outcomes else 'conflicting' if len(outcomes) > 1 else 'yes' if True in outcomes else 'no'
            counterexamples = [t['id'] for t in trials if rule['expected'] is not None and t['outcome'] != rule['expected']]
            bands.append({**deepcopy(rule), 'height': max_depth-rule['depth'],
                          'status': status, 'trial_ids': [t['id'] for t in trials],
                          'unknown_context_ids': unknown, 'counterexample_ids': counterexamples,
                          'historical_parent': bool(rule['children']),
                          'colour_labels': {k: self._factors[k] for k in rule['conditions']}})
            if rule['children']:
                continue  # Parent conflicts remain visible, but branches drive questions.
            pending = []
            if not trials:
                pending.append(('untested_case', 'Obtain an observed trial for this condition combination.'))
            elif rule['expected'] is None:
                pending.append(('unmodelled_relationship', 'Propose a condition-to-outcome rule and a test; observations alone do not specify one.'))
            if counterexamples or len(outcomes) > 1:
                pending.append(('unexplained_observation', 'Which additional condition explains these counterexamples?'))
            if unknown:
                pending.append(('missing_context', 'Can the missing condition be measured for these trials? Do not guess it.'))
            for kind, question in pending:
                gaps.append(dict(id=f"{self._identity['id']}:{rule['id']}:{kind}",
                                 rule_id=rule['id'], kind=kind, question=question,
                                 conditions=deepcopy(rule['conditions']),
                                 scope=self._identity['scope'], revision=self._revision,
                                 subject=self._identity['subject'], action=self._identity['action'],
                                 target_question=self._identity['question'],
                                 counterexample_ids=counterexamples.copy(),
                                 unknown_context_ids=unknown.copy(), execution_ready=False))
        bundles = [dict(observation_id=t['id'], bands=[
            dict(factor=name, colour=colour, height=i,
                 state='unknown' if name not in t['bits'] else 'yes' if t['bits'][name] else 'no')
            for i, (name, colour) in enumerate(self._factors.items())])
            for t in self._trials.values()]
        return deepcopy({**self._identity, 'revision': self._revision,
                         'persistent': True, 'trial_count': len(self._trials),
                         'factor_colours': self._factors, 'bands': bands, 'gaps': gaps,
                         'trials': list(self._trials.values()), 'bit_bundles': bundles, 'arrivals': self._arrivals,
                         'refinements': self._events, 'execution_authorized': False,
                         'meaning': 'observed coverage and candidate rules; not causal verification'})


In [ ]:
%%writefile /kaggle/working/my_agent.py
"""Submission experiment based on the exact reviewed notebook agent.

Archive/conditional frame-change learning runs from observed transitions only.
The policy experiment uses consistent observed displacement in the BFS planner.
Frame-change hypotheses and retrieved archives do NOT authorize game actions.
"""
from collections import deque
from pathlib import Path
from uuid import uuid4

from prioragent_reviewed_baseline import PriorAgent as ReviewedPriorAgent, _grid, GameAction, GameState
from learning_session import LearningSession
from orbital_archive import OrbitalArchive


class PriorAgent(ReviewedPriorAgent):
    CANDIDATE_VERSION = 'orbital-learning-1'

    def __init__(self, *args, **kwargs):
        archive_path = kwargs.pop('archive_path', None)
        self.learned_planning = kwargs.pop('learned_planning', True)
        super().__init__(*args, **kwargs)
        self._instance = uuid4().hex
        archive_path = archive_path or Path('learning-archives') / (self._instance+'.sqlite3')
        self._archive = OrbitalArchive(archive_path)
        self.learning = LearningSession(self._archive, self._instance)
        self._issued, self._issued_data, self._capture_counter = None, {}, None
        self.learned_plan_calls = 0
        self.learning_summary = {}

    def _capture(self, latest_frame):
        counter = getattr(self, 'action_counter', len(self.learning.events))
        if counter == self._capture_counter:
            return
        g = _grid(latest_frame)
        if g is None or g.ndim != 2 or not g.size:
            return
        result = self.learning.observe(dict(observation_id=f'{self._instance}:{counter}',
            game_id=str(self.game_id), instance_id=self._instance,
            level=int(latest_frame.levels_completed), state=getattr(latest_frame.state, 'name', latest_frame.state),
            frame=g.tolist(), previous_action=self._issued, previous_data=self._issued_data))
        self._capture_counter = counter
        if result.get('event') in ('reset', 'death', 'win', 'level_change', 'game_change', 'session_change', 'uncertain_boundary'):
            # A boundary is not the ordinary effect of the previous command.
            self.prev_grid = self.prev_action = self.prev_blob = None
            self.plan.clear()
            if result['event'] in ('level_change', 'game_change', 'session_change', 'uncertain_boundary'):
                for effect in self.effects.values():
                    effect['dy'].clear(); effect['dx'].clear()

    def choose_action(self, frames, latest_frame):
        self._capture(latest_frame)
        action = super().choose_action(frames, latest_frame)
        self._issued = int(action.value)
        self._issued_data = dict(getattr(action, 'data', {}) or {})
        if self._issued == 6 and getattr(self, '_last_click', None):
            self._issued_data = dict(x=self._last_click[0], y=self._last_click[1])
        if self._issued == 0:
            self.prev_grid = self.prev_action = self.prev_blob = None
            self.plan.clear()
        return action

    def _movement_deltas(self, q):
        deltas = {1: (-q, 0), 2: (q, 0), 3: (0, -q), 4: (0, q)}
        if not self.learned_planning:
            return deltas
        for action in deltas:
            effect = self.effects.get(action, {})
            dys, dxs = effect.get('dy', []), effect.get('dx', [])
            if len(dys) < 3 or len(dys) != len(dxs):
                continue
            moves = {(int(dy), int(dx)) for dy, dx in zip(dys[-4:], dxs[-4:])}
            if len(moves) != 1:
                continue
            dy, dx = next(iter(moves))
            if (dy == 0) != (dx == 0) and 0 < max(abs(dy), abs(dx)) <= 2*q:
                deltas[action] = (dy, dx)
        return deltas

    def _bfs_step(self, g, start, goal, q):
        deltas = self._movement_deltas(q)
        priors = {1: (-q, 0), 2: (q, 0), 3: (0, -q), 4: (0, q)}
        if deltas == priors:
            return super()._bfs_step(g, start, goal, q)
        self.learned_plan_calls += 1
        r = max(1, q//2)
        queue, seen = deque([(start, None)]), {start}
        while queue:
            position, first = queue.popleft()
            if abs(position[0]-goal[0])+abs(position[1]-goal[1]) <= q:
                return first
            for action, (dy, dx) in deltas.items():
                target = position[0]+dy, position[1]+dx
                if target in seen or not self._passable(g, target[0], target[1], r):
                    continue
                seen.add(target)
                queue.append((target, action if first is None else first))
        return None

    def main(self):
        try:
            return super().main()
        finally:
            try:
                if self.arc_env and self.arc_env.observation_space:
                    self._capture(self._convert_raw_frame_data(self.arc_env.observation_space))
                self.learning.archive_current('run_end')
                self.learning_summary = dict(observations=len(self.learning.events),
                    bundles=len(self.learning.closed_bundles), refinements=len(self.learning.refinements),
                    learned_plan_calls=self.learned_plan_calls, archive=str(self._archive.path))
            finally:
                self._archive.close()


In [ ]:
import hashlib
from pathlib import Path
SOURCE_HASHES = {'prioragent_reviewed_baseline.py': '6c8a9500ad7a8035306e8ff83c712140333465b6d7c98440845ba1aef4271318', 'orbital_archive.py': '9c5286f7786283ec1a1d707074b363062d9a13e1327defcf20444c9f96e0c3c8', 'learning_session.py': '83af0d156ae8f4ab12f136f9db5cc156d625302db98a4009ab0789a55b43d232', 'automatic_pin_refiner.py': 'd3ce9850ab896b5baf0cfdc44944a930a9bb21135a7bc84a78b375fe608f297a', 'evidence_pin.py': 'e4876e4ee534c9bd0301437ef4e7cd4e6ff2562d910c49e1c79860a5fb219c90', 'my_agent.py': 'ab639e3859896f9bda61702c90bda35142959db3a95222e2700207804b8060b5'}
for name, expected in SOURCE_HASHES.items():
    actual = hashlib.sha256((Path('/kaggle/working') / name).read_bytes()).hexdigest()
    assert actual == expected, f'Source mismatch: {name}'


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for gateway to be ready
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy repo to writable location
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    import shutil
    for name in ('prioragent_reviewed_baseline.py', 'orbital_archive.py', 'learning_session.py', 'automatic_pin_refiner.py', 'evidence_pin.py'):
        shutil.copy2('/kaggle/working/' + name, '/kaggle/working/ARC-AGI-3-Agents/' + name)

    # Install our agent
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write('''from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import PriorAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "prioragent": PriorAgent,
}
''')

    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write('''SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
''')

    # Run the agent against every hidden game
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent prioragent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Non-competition mode: produce a dummy submission
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()
